# TP1 : Connexion à un canal

**Contexte :** assistant du service client de Lagune Banque.

Vous allez compléter un adaptateur WhatsApp et le relier au cœur de l'agent par la passerelle, sans modifier ni l'un ni l'autre. Les tests se font sans réseau.

```mermaid
flowchart LR
    A["Client"] -->|"message du webhook"| B["CanalWhatsAppTP.analyser"]
    B -->|"Message"| C["Passerelle.traiter"]
    C -->|"reply"| D["Assistant (coeur.py)"]
    D -->|"Reponse"| C
    C --> E["CanalWhatsAppTP.envoyer"]
    E --> A
```

Complétez les `# TODO` de `service/canaux/whatsapp_tp.py`.

## <a id="configuration"></a>Configuration

In [ ]:
import logging
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [ ]:
import json
import os
import sys
from pathlib import Path

import litellm
from dotenv import load_dotenv

# Le notebook peut être lancé depuis la racine du dépôt ou depuis son propre dossier : on cherche le .env.
for dossier in [Path.cwd(), *Path.cwd().parents]:
    if (dossier / ".env").exists():
        load_dotenv(dossier / ".env")
        break

In [ ]:
MODELE = os.getenv("MODELE")
API_KEY = os.getenv("LLM_API_KEY")
litellm.drop_params = True  # ignore les paramètres que le modèle ne supporte pas

if not (MODELE and API_KEY):
    raise RuntimeError("Renseignez LLM_API_KEY et MODELE dans .env.")
try:
    litellm.completion(model=MODELE, api_key=API_KEY, max_tokens=20, messages=[{"role": "user", "content": "Réponds : OK"}])
    print("Connexion OK :", MODELE)
except Exception as e:
    raise RuntimeError(f"Connexion impossible ({type(e).__name__}). Vérifiez MODELE et LLM_API_KEY.") from None

In [ ]:
# Le code commun du module 5 : cœur de l'agent, garde-fous, suivi, passerelle (socle/) et canaux (service/)
RACINE_M5 = next(p for p in [Path("05_canaux_gouvernance"), Path("."), Path(".."), Path("../.."), Path("../../05_canaux_gouvernance")]
                 if (p / "socle" / "coeur.py").exists()).resolve()
sys.path.insert(0, str(RACINE_M5 / "socle"))
sys.path.insert(0, str(RACINE_M5 / "service"))
from outils import AUJOURDHUI, KOUASSI, SYSTEM_PROMPT, reset_db, sql
import garde_fous, suivi
from coeur import Assistant, setup_db
from passerelle import Message, Passerelle

reset_db()

## <a id="ladaptateur"></a>L'adaptateur

Ouvrez `service/canaux/whatsapp_tp.py` et complétez TODO 1 (`analyser`) et TODO 2 (`envoyer`).

In [ ]:
from canaux import whatsapp_tp
import importlib
importlib.reload(whatsapp_tp)
print(whatsapp_tp.__file__)

## <a id="tester-ladaptateur-sans-reseau"></a>Tester l'adaptateur sans réseau

La cellule suivante donne à l'adaptateur un message au format du webhook WhatsApp, puis intercepte l'appel HTTP sortant. Vous voyez exactement ce que l'API de Meta recevrait.

In [ ]:
import httpx
from unittest.mock import patch


class FausseReponse:
    status_code, text = 200, "ok"


envois = []                                              # (url, json) de chaque message « envoyé »

async def faux_post(self, url, json=None, **kw):
    envois.append((url, json))
    return FausseReponse()


canal = whatsapp_tp.CanalWhatsAppTP("TOKEN", "1234567890")
recu = {"from": "2250700000000", "id": "wamid.12", "type": "text", "text": {"body": "Bonjour"}}
msg = canal.analyser(recu)
print("Message :", msg)
print("Sans texte :", canal.analyser({"from": "2250700000000", "id": "wamid.13", "type": "image"}))

with patch.object(httpx.AsyncClient, "post", faux_post):
    await canal.envoyer("2250700000000", "Bonjour, ceci est un test.")
    await canal.envoyer("2250700000000", "x" * 5000)     # plus long que la limite de WhatsApp
print([(u.split("/")[-1], j["to"], len(j["text"]["body"])) for u, j in envois])
envois_test = list(envois)             # copie pour le contrôle final

## <a id="echange-complet"></a>Échange complet

Un faux client écrit à l'assistant. Votre adaptateur analyse le message, la passerelle identifie le client, le cœur répond, puis votre adaptateur envoie la réponse. Le cœur n'est pas modifié.

Pour chaque message, le cœur appelle le modèle avec deux prompts du socle : `PROMPT_CLASSEMENT` choisit le parcours (P1 à P5), puis `SYSTEM_PROMPT` rédige la réponse sur l'échéance (parcours P2). La cellule suivante les affiche.

In [ ]:
from coeur import PROMPT_CLASSEMENT

print(PROMPT_CLASSEMENT)
print("\n" + "-" * 100 + "\n")
print(SYSTEM_PROMPT)

In [ ]:
envois.clear()
reset_db()
passerelle = Passerelle(Assistant(), {"whatsapp": canal})


async def client_ecrit(texte, n):
    recu = {"from": "2250711111111", "id": f"wamid.{n}", "type": "text", "text": {"body": texte}}
    with patch.object(httpx.AsyncClient, "post", faux_post):
        await passerelle.traiter(canal.analyser(recu))
    return envois[-1][1]["text"]["body"]


for n, texte in enumerate(["Quand est ma prochaine échéance ?", "/client CLI-0001", "Quand est ma prochaine échéance ?",
                           "Pouvez-vous m'accorder un prêt ?"], start=1):
    print(f"CLIENT    : {texte}\nASSISTANT : {(await client_ecrit(texte, n))[:260]}\n")

Points à observer :

- Tant que le client n'est pas identifié, la passerelle répond par la consigne d'identification. Un canal n'authentifie pas un client bancaire : ici, `/client` sert à s'identifier.
- La première réponse annonce que l'interlocuteur est un assistant virtuel.
- Le garde-fou bloque la demande de crédit.

## <a id="bonus-second-canal"></a>Bonus : second canal

Un canal doit fournir deux éléments : un attribut `nom` et une méthode `envoyer`. La cellule suivante définit un canal qui écrit dans la console et l'ajoute à la passerelle à côté du premier, sans toucher à l'agent.

In [ ]:
class CanalConsole:
    nom = "console"

    async def envoyer(self, utilisateur, texte):
        print(f"[console → {utilisateur}] {texte[:200]}")


passerelle2 = Passerelle(passerelle.assistant, {"whatsapp": canal, "console": CanalConsole()})
await passerelle2.traiter(Message("console", "terminal-1", "/client CLI-0001"))
await passerelle2.traiter(Message("console", "terminal-1", "Quand est ma prochaine échéance ?"))

## Contrôle automatique

La dernière cellule affiche ÉCHEC tant que les `# TODO` ne sont pas faits.

In [ ]:
try:
    assert msg is not None and msg.canal == "whatsapp" and msg.utilisateur == "2250700000000" and msg.texte == "Bonjour" and msg.id == "wamid.12", \
        "analyser : Message('whatsapp', from, texte, id) (TODO 1)"
    assert canal.analyser({"from": "2250700000000", "id": "wamid.13", "type": "image"}) is None, "analyser : None si le type n'est pas text (TODO 1)"
    assert envois_test and all(u.endswith("/messages") for u, _ in envois_test), "envoyer : POST sur self.url (TODO 2)"
    assert all(j["to"] == "2250700000000" for _, j in envois_test), "envoyer : `to` doit être le numéro de l'utilisateur (TODO 2)"
    assert max(len(j["text"]["body"]) for _, j in envois_test) <= 4096 and sum(len(j["text"]["body"]) for _, j in envois_test) == 5000 + len("Bonjour, ceci est un test."),         "envoyer : découper les messages de plus de 4096 caractères, sans rien perdre (TODO 2)"
    print("OK")
except Exception as e:
    print(f"ÉCHEC : {e if isinstance(e, AssertionError) else type(e).__name__ + ' - ' + str(e)}")